# Queue: augmentation study, scratchcheck — 3 runs

Feasibility check: the TrOCR-small architecture trained FROM SCRATCH (`train_trocrscratch.ipynb`, random init, 300 epochs, ~1.5-2 h each), seed 42, on `none`, `geom2` and `elastic4`. Go on to all 17 conditions only if the CER is usable (roughly under 20-25%) and the three conditions clearly differ.

Each run pushes to its own public HF repo `cyttic/hmt-<model>-<cond>-s<seed>`; a run whose repo already holds `results.json` is skipped, so re-running resumes the queue. A failed run's executed notebook goes to the public HF dataset `cyttic/a5000-queue-logs`. The local run folder is deleted after a successful push.

Start on the server: `./run.sh queue_scratchcheck.ipynb` (inside tmux).

In [ ]:
import os, json, time, traceback, shutil
import papermill as pm
from huggingface_hub import HfApi, hf_hub_download
WAVE = "scratchcheck"
RUNS = [["trocrscratch", "none", 42], ["trocrscratch", "geom2", 42], ["trocrscratch", "elastic4", 42]]   # (model, condition, seed)
NB = {"trocrctc": "train_trocrctc.ipynb", "trocrsmall": "train_trocrsmall.ipynb", "htrvt": "train_htrvt.ipynb", "trocrscratch": "train_trocrscratch.ipynb",
      "trocrscratchreg": "train_trocrscratchreg.ipynb"}
HERE = os.getcwd(); BASE = os.path.expanduser(f"~/work/hmt_{WAVE}"); os.makedirs(BASE, exist_ok=True)
api = HfApi(); LOGS = "cyttic/a5000-queue-logs"   # PUBLIC dataset: executed notebooks of failed runs (tracebacks)
api.create_repo(LOGS, repo_type="dataset", private=False, exist_ok=True)
def repo(m, c, s): return f"cyttic/hmt-{m}-{c}-s{s}"
def done(r):
    try: return "results.json" in api.list_repo_files(r)
    except Exception: return False
def push_log(path, name):
    try: api.upload_file(path_or_fileobj=path, path_in_repo=f"{name}.ipynb", repo_id=LOGS, repo_type="dataset", commit_message=f"failed run {name}"); print("log ->", f"{LOGS}/{name}.ipynb", flush=True)
    except Exception as e: print("log upload failed", repr(e)[:200], flush=True)
status = {}
for i, (m, c, s) in enumerate(RUNS, 1):
    r, name = repo(m, c, s), f"hmt_{m}_{c}_s{s}"
    if done(r): print(f"[{i}/{len(RUNS)}] {name}: already on HF, skipped", flush=True); status[name] = "skipped (done)"; continue
    wd = os.path.join(BASE, name); os.makedirs(wd, exist_ok=True); t0 = time.time()
    print(f"[{i}/{len(RUNS)}] {name} -> {r}  started {time.strftime('%H:%M')}", flush=True)
    try:
        pm.execute_notebook(os.path.join(HERE, NB[m]), os.path.join(wd, name + ".ipynb"), cwd=wd, log_output=True, kernel_name="python3",
                            parameters={"COND": c, "SEED": s, "REPO_DIR": HERE})
        status[name] = f"ok ({(time.time() - t0) / 3600:.2f} h)" if done(r) else "finished, but results.json not on HF"
        if done(r): shutil.rmtree(wd, ignore_errors=True)   # pushed: free the disk
    except Exception as e:
        status[name] = "FAILED: " + repr(e)[:300]; traceback.print_exc(); push_log(os.path.join(wd, name + ".ipynb"), f"hmt_{WAVE}__" + name)
    print(f"[{i}/{len(RUNS)}] {name}: {status[name]}", flush=True)
print("\n==== queue summary"); [print(f"{k:45s} {v}") for k, v in status.items()]

## Results on HF

In [ ]:
for m, c, s in RUNS:
    r = repo(m, c, s)
    if not done(r): print(f"{r}: no results"); continue
    T = json.load(open(hf_hub_download(r, "results.json", force_download=True)))["final"]["test_greedy"]
    print(f"{r:44s} CER {T['cer']:.4f} | WER {T['wer']:.4f}")